In [1]:
import pickle
import numpy as np
from math import log
import networkx as nx
import inspect
from netective.structure import properties
parent_class = properties._Property

from netective.structure.structure import Structure, characterize_network

In [2]:
def synthetic_charact(num_nodes, density, random_times):

    child_classes = {}
    for name, obj in inspect.getmembers(properties):
        if inspect.isclass(obj) and issubclass(obj, parent_class) and obj != parent_class:
            if obj._use_direction:
                continue
            bool_mask = [
                    obj._use_direction,
                    obj._use_selfloops,
                    obj._use_giant_component,
                    obj._use_paths,
                ]
            child_classes[obj] = np.packbits(bool_mask).item() >> 4

    nodes = {}
    edges = {}
    scalar_raw = {}
    scalar_networknorm = {}
    k = int(round(density*num_nodes / 2))    # Watts-Strogatz k, Barabasi-Albert m, number of edges to attach from a new node to existing nodes
    k = 2 if k < 2 else k # k must be at least 2 for WS
    dgm_gen_n = int(round(log(2/3, 3) + log(num_nodes,3) + 1))

    for i in range(random_times):
        networks = {}
        networks['ba_graph'] = nx.barabasi_albert_graph(n=num_nodes, m=k, seed=i)
        networks['er_graph'] = nx.erdos_renyi_graph(n=num_nodes, p=density, seed=i)
        networks['ws_graph'] = nx.watts_strogatz_graph(n=num_nodes, k=k, p=0.5, seed=i)
        networks['dgm_graph'] = nx.dorogovtsev_goltsev_mendes_graph(n=dgm_gen_n) # already deterministic
        networks['sf_graph'] = nx.Graph(nx.scale_free_graph(n=num_nodes, seed=i)) # original scale-free graph is MultiGraph


        for name, network in networks.items():

            nodes[f'{name}_{density}_{num_nodes}_{i}'] = network.number_of_nodes()
            edges[f'{name}_{density}_{num_nodes}_{i}'] = network.number_of_edges()

            struct = Structure(network, norm=None, net_id=name, verbose=False)
            scalar_values, dist_props = struct.get_props(child_classes=child_classes)
            scalar_raw[f'{name}_{density}_{num_nodes}_{i}'] = scalar_values[name]
            for prop_name, prop_values in dist_props[name].items():
                scalar_raw[f'{name}_{density}_{num_nodes}_{i}'][f'Average {prop_name}'] = prop_values[0]
                scalar_raw[f'{name}_{density}_{num_nodes}_{i}'][f'Variation {prop_name}'] = prop_values[1]
                scalar_raw[f'{name}_{density}_{num_nodes}_{i}'][f'Skewness {prop_name}'] = prop_values[2]
                scalar_raw[f'{name}_{density}_{num_nodes}_{i}'][f'Kurtosis {prop_name}'] = prop_values[3]

            struct.norm = 'network'
            scalar_values, dist_props = struct.get_props(child_classes=child_classes)
            scalar_networknorm[f'{name}_{density}_{num_nodes}_{i}'] = scalar_values[name]
            for prop_name, prop_values in dist_props[name].items():
                scalar_networknorm[f'{name}_{density}_{num_nodes}_{i}'][f'Average {prop_name}'] = prop_values[0]
                scalar_networknorm[f'{name}_{density}_{num_nodes}_{i}'][f'Variation {prop_name}'] = prop_values[1]
                scalar_networknorm[f'{name}_{density}_{num_nodes}_{i}'][f'Skewness {prop_name}'] = prop_values[2]
                scalar_networknorm[f'{name}_{density}_{num_nodes}_{i}'][f'Kurtosis {prop_name}'] = prop_values[3]
    
    data_dict = {
        f'scalar_networknorm_{density}_{num_nodes}' : scalar_networknorm,
        f'scalar_raw_{density}_{num_nodes}' : scalar_raw,
        f'nodes_{density}_{num_nodes}' : nodes,
        f'edges_{density}_{num_nodes}' : edges
    }
                 
    return data_dict

In [3]:
random_times = 1  # number of times to run random graph generation
num_nodes_values = range(1000, 5001, 1000)
density_values = [0.001, 0.005, 0.01, 0.05]

input_datasets = [(num_nodes, density, random_times) for num_nodes in num_nodes_values for density in density_values]
input_datasets[:5]

[(1000, 0.001, 1),
 (1000, 0.005, 1),
 (1000, 0.01, 1),
 (1000, 0.05, 1),
 (2000, 0.001, 1)]

In [4]:
synthetic_charact(1000, 0.001, 1)

c:\Users\jmere\.conda\envs\pyfl38\lib\site-packages\networkx\algorithms\centrality\subgraph_alg.py:182: RuntimeWarning: overflow encountered in exp
  expw = np.exp(w)
C:\Users\jmere\Dropbox (FreyreLab)\netective\src\netective\structure\properties.py:959: RuntimeWarning: invalid value encountered in scalar divide
  self._raw_value.append(n_int / (n_int + n_ext))


{'scalar_networknorm_0.001_1000': {'ba_graph_0.001_1000_0': {'Average Local Efficiency': 0.032146692013854096,
   'Average Shortest Path Length': 0.00410059308557807,
   'Center': 0.001,
   'Diameter': 0.007007007007007007,
   'Global Efficiency': 0.26042448162448156,
   'Periphery': 0.175,
   'Radius': 0.004004004004004004,
   'Entropy of Degree Distribution': 0.2503187570778676,
   'Max Degree': 0.1,
   'Self-Loops': 0.0,
   'Undirected Gini Index': 0.3807765531062123,
   'Gene % in the Giant Component': 1.0,
   'Undirected Density': 0.001996,
   'Average Average Degree for Nearest Neighbors (Undirected)': 0.012076861352842574,
   'Variation Average Degree for Nearest Neighbors (Undirected)': 0.00012303244633254481,
   'Skewness Average Degree for Nearest Neighbors (Undirected)': 2.5085499728178666,
   'Kurtosis Average Degree for Nearest Neighbors (Undirected)': 7.1917524088461136,
   'Average Clustering Coefficient': 0.03174579347017994,
   'Variation Clustering Coefficient': 0.021

In [110]:
from tqdm import tqdm
import concurrent.futures

def run_parallel(f, my_iter, workers):
    len_iter = len(my_iter)
    with tqdm(total=len_iter) as pbar:
        with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
            futures = {}
            for arg in my_iter:
                print(f'Running: {arg[0]} nodes, {arg[1]} density, {arg[2]} random times each model...')
                name = f'{arg[0]}_{arg[1]}'
                futures[executor.submit(f, *arg)] = name

            results = {}
            for future in concurrent.futures.as_completed(futures):
                try:
                    results[name] = future.result()
                    pbar.update(1)
                except Exception as exc:
                    print(f"Error: {exc}")


    return results

In [112]:
for name, data_dict in run_parallel(synthetic_charact, input_datasets, 10).items():
        for name, data in data_dict.items():
            with open(f'{name}.pkl', 'wb') as f:
                pickle.dump(data, f)

  0%|          | 0/20 [00:00<?, ?it/s]

Running: 1000 nodes, 0.001 density, 1 random times each model...
Running: 1000 nodes, 0.005 density, 1 random times each model...
Running: 1000 nodes, 0.01 density, 1 random times each model...
Running: 1000 nodes, 0.05 density, 1 random times each model...
Running: 2000 nodes, 0.001 density, 1 random times each model...
Running: 2000 nodes, 0.005 density, 1 random times each model...
Running: 2000 nodes, 0.01 density, 1 random times each model...
Running: 2000 nodes, 0.05 density, 1 random times each model...
Running: 3000 nodes, 0.001 density, 1 random times each model...
Running: 3000 nodes, 0.005 density, 1 random times each model...
Running: 3000 nodes, 0.01 density, 1 random times each model...
Running: 3000 nodes, 0.05 density, 1 random times each model...
Running: 4000 nodes, 0.001 density, 1 random times each model...
Running: 4000 nodes, 0.005 density, 1 random times each model...
Running: 4000 nodes, 0.01 density, 1 random times each model...
Running: 4000 nodes, 0.05 densit

In [113]:
# open pickle file
with open('edges_0.001_2000.pkl', 'rb') as f:
    scalar_raw = pickle.load(f)

scalar_raw

{'ba_graph_0.001_2000_0': 3996,
 'er_graph_0.001_2000_0': 2071,
 'ws_graph_0.001_2000_0': 2000,
 'dgm_graph_0.001_2000_0': 6561,
 'sf_graph_0.001_2000_0': 3281,
 'ba_graph_0.001_2000_1': 3996,
 'er_graph_0.001_2000_1': 1974,
 'ws_graph_0.001_2000_1': 2000,
 'dgm_graph_0.001_2000_1': 6561,
 'sf_graph_0.001_2000_1': 3214}